# Clean up an experiment's results on Drive

Deletes what a sweep notebook wrote to Drive. Each notebook writes to
`MyDrive/<notebook name>/` - `naive-beam-search-colab.ipynb` derives that folder
from its own filename, so experiments never share one - and six files per model
underneath.

**This notebook cannot detect the target for you.** It would only ever find its
own name, which is not the folder you want emptied. Section 1 lists what is in
Drive; set `EXPERIMENT` in section 2 to the one you mean.

**The weights were never here.** They stay on local Colab scratch on purpose -
~127 GiB would exhaust the Drive quota - and `run_one()` deletes each model in a
`finally`. So these folders hold CSVs and JSON, a few MB per model, not the big
download.

**Section 4 is not reversible.** Deleting through the mounted Drive filesystem
does not reliably land in Drive's Trash the way a deletion from
drive.google.com does. Read section 3 first; section 4 refuses to delete
anything until you set `CONFIRM = True` by hand.

## 1. Mount Drive, and see which experiments are there

Every folder in `MyDrive` that holds a `summary-row.json` underneath it was
written by one of these sweeps.

In [ ]:
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

MYDRIVE = Path("/content/drive/MyDrive")

candidates = sorted(
    folder for folder in MYDRIVE.iterdir()
    if folder.is_dir() and any(folder.rglob("summary-row.json"))
)

if not candidates:
    print("no experiment folders found in MyDrive")
else:
    print("experiment folders in MyDrive:\n")
    for folder in candidates:
        files = [p for p in folder.rglob("*") if p.is_file()]
        size = sum(p.stat().st_size for p in files)
        models = sum(1 for _ in folder.glob("*/summary-row.json"))
        print(f"  {folder.name:<40} {models} models, "
              f"{len(files)} files, {size / 1e6:.2f} MB")

## 2. Pick the one to delete

Copy a name from the listing above. Nothing is deleted by naming it - sections 3
and 4 still have to run.

In [ ]:
# The folder to clean, exactly as it appears above.
EXPERIMENT = "naive-beam-search-colab"

RESULTS_ROOT = MYDRIVE / EXPERIMENT

print(f"target   {RESULTS_ROOT}")
print(f"exists   {RESULTS_ROOT.exists()}")

## 3. What is actually in it

Read this before running section 4. If the sweep never finished a model, or an
earlier cleanup already ran, this prints nothing and there is nothing to do.

In [ ]:
files = []

if not RESULTS_ROOT.exists():
    print("nothing on Drive - no earlier run wrote here, or it is already gone")
else:
    files = sorted(p for p in RESULTS_ROOT.rglob("*") if p.is_file())

    by_folder = {}
    for path in files:
        by_folder.setdefault(path.parent, []).append(path)

    for folder in sorted(by_folder):
        label = folder.relative_to(RESULTS_ROOT)
        subtotal = sum(p.stat().st_size for p in by_folder[folder])
        name = "(root)" if str(label) == "." else str(label)
        print(f"{name}   {len(by_folder[folder])} files, {subtotal / 1e6:.2f} MB")
        for path in by_folder[folder]:
            print(f"    {path.stat().st_size:>12,}  {path.name}")

    total = sum(p.stat().st_size for p in files)
    print()
    print(f"{len(files)} files in {len(by_folder)} folders, {total / 1e6:.2f} MB total")

## 4. Delete

Set `CONFIRM = True` only once the listing above is what you expect to lose.
Left `False`, this is a dry run that names every file it would remove.

In [ ]:
import shutil

CONFIRM = False        # <- set True to actually delete

if not RESULTS_ROOT.exists():
    print("nothing to delete")
elif not CONFIRM:
    print(f"DRY RUN - would delete {len(files)} files and the folder itself:")
    print()
    for path in files:
        print(f"  {path.relative_to(RESULTS_ROOT)}")
    print()
    print("set CONFIRM = True in this cell to do it")
else:
    shutil.rmtree(RESULTS_ROOT)
    print(f"deleted {RESULTS_ROOT}")

## 5. Verify

In [ ]:
if RESULTS_ROOT.exists():
    remaining = [p for p in RESULTS_ROOT.rglob("*") if p.is_file()]
    print(f"still present: {len(remaining)} files under {RESULTS_ROOT}")
else:
    print(f"gone: {RESULTS_ROOT} no longer exists")

## Not covered here

Neither is written by the sweep, so neither is touched above - check them by
hand if you want a clean slate:

- `MyDrive/Colab Notebooks/` - a copy of the experiment notebook, if you
  uploaded one rather than opening it from GitHub.
- Your local Downloads folder - `<experiment>-results.zip`, which the sweep's
  last cell pushes through `files.download()`.